## Sistema alemán y francés

In [1]:
import pandas as pd

In [2]:
# ------------------------------------------------------------------
# 0. SUPUESTOS (cámbialos si tu enunciado dice otra cosa)
# ------------------------------------------------------------------
FINANCIAR_PRIMA = False  # True: capital = Monto_Inicial + Prima
TIPO = "nominal"  # "nominal": i = tipo/12 | "TAE": i = (1+tipo)^(1/12)-1

# ------------------------------------------------------------------
# 1. DATOS
# ------------------------------------------------------------------
df_filtrado = pd.read_excel('datos\datos_automovil.xlsx')
df = df_filtrado.copy()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df = df.reset_index(drop=True)

print("Préstamos:", len(df))
print("Nulos:", int(df.isna().sum().sum()))

# ------------------------------------------------------------------
# 2. VARIABLES COMUNES
# ------------------------------------------------------------------
P = df["Monto_Inicial"].astype(float) + (df["Prima"] if FINANCIAR_PRIMA else 0)
n = df["Duracion"].astype(int)  # meses
tipo_anual = df["Ratio_Interes"] / 100  # 19.7 -> 0.197
i = tipo_anual / 12 if TIPO == "nominal" else (1 + tipo_anual) ** (1 / 12) - 1

df["Capital"] = P

# ------------------------------------------------------------------
# 3. SISTEMA FRANCÉS — cuota constante
# ------------------------------------------------------------------
df["Cuota_Fr"] = P * i / (1 - (1 + i) ** -n)
df["Intereses_Fr"] = df["Cuota_Fr"] * n - P

# ------------------------------------------------------------------
# 4. SISTEMA ALEMÁN — capital amortizado constante, cuota decreciente
# ------------------------------------------------------------------
df["Amort_De"] = P / n
df["Cuota1_De"] = P / n + P * i    # primera cuota (la más alta)
df["Intereses_De"] = P * i * (n + 1) / 2

# ------------------------------------------------------------------
# 5. COMPARATIVA Y DECISIÓN
# ------------------------------------------------------------------
resumen = pd.DataFrame({
    "Francés": [df.Intereses_Fr.sum(), df.Cuota_Fr.mean()],
    "Alemán":  [df.Intereses_De.sum(), df.Cuota1_De.mean()],
}, index=["Intereses totales", "Cuota inicial media"])

pd.options.display.float_format = "{:,.2f}".format
print("\nCOMPARATIVA:\n", resumen)

# Decisión automática: el sistema con menos intereses totales pagados
METODO = "Francés" if df.Intereses_Fr.sum() < df.Intereses_De.sum() else "Alemán"
print(f"\nMétodo elegido para continuar: {METODO}")

<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
C:\Users\agara\AppData\Local\Temp\ipykernel_26280\269730073.py:10: SyntaxWarning: invalid escape sequence '\d'
  df_filtrado = pd.read_excel('datos\datos_automovil.xlsx')


Préstamos: 76925
Nulos: 0

COMPARATIVA:
                            Francés         Alemán
Intereses totales   625,855,099.45 585,621,492.66
Cuota inicial media       1,690.50       1,870.29

Método elegido para continuar: Alemán
